In [133]:
import glob
import tqdm
import sleap_io as sio
import pandas as pd
import numpy as np
import os
import re

In [134]:
experiment_no = [384]
excluded_indices = []

In [135]:
# Functions to sort file names correctly
def atoi(text):
    return int(text) if text.isdigit() else text

def natural_keys(text):
    return [atoi(c) for c in re.split(r'(\d+)', text)]

#### Making CSV from slp files

In [136]:
for exp in experiment_no:
    folder = f"D:/big_setup/experiment_{exp}/concatenated_data_cam_mic_sync/"
    labels_path = folder+"sleap_predictions/"
    tracks_path = folder+"tracks/"
    os.makedirs(tracks_path,exist_ok=True)

    labels = glob.glob(labels_path+"*.slp")
    for label in tqdm.tqdm(labels):
        preds = sio.load_file(label)
        tmp_str = label.split("\\")[-1].split(".")[0]
        sio.save_csv(preds,tracks_path+"labels.body_1_anchor."+tmp_str+".analysis.csv",save_metadata= True)
        # data = sio.load_csv(tracks_path+"labels.body_1_anchor."+tmp_str+".analysis.csv")
        # data.save(tracks_path+"labels.body_1_anchor."+tmp_str+".analysis.csv.slp")
        
    print(f"Experiment {exp} done")


100%|██████████| 7/7 [00:02<00:00,  2.84it/s]

Experiment 384 done


#### Finding the distances between each node

In [137]:
# skeleton
# nose to head = 40
# ear_l to head = 14
# ear_r to head = 14
# head to neck = 10
# neck to body_1 = 20
# body_1 to body_2 = 22
# body_2 to body_3 = 22


# Found the above using the bottom code - I used experiment 384 for this

"""
def avg_euc_dist(df,point_1,point_2):
     return pd.Series(np.linalg.norm(df[[f'{point_1}.x',f'{point_1}.y']].values - df[[f'{point_2}.x',f'{point_2}.y']].values, axis =1 ))


folder = f"D:/big_setup/experiment_{experiment_no[0]}/concatenated_data_cam_mic_sync/"
labels_path = folder+"tracks/"
files = glob.glob(labels_path+"*.csv")
for file in files:
     temp = avg_euc_dist(pd.read_csv(file),'body_2','body_3')
     bins =  pd.qcut(range(20,40), 29,retbins = True,duplicates='drop')[1]
     temp_2 = pd.cut(temp, bins=bins, right=True, include_lowest=True)
     bin_counts = temp.groupby(temp_2).size() 

     mode_bin = bin_counts.idxmax()
     mode_count = bin_counts.max()
     print(temp.mean())
     print(mode_bin,mode_count)
     # plt.figure()
     # temp.hist(bins =bins)
"""


'\ndef avg_euc_dist(df,point_1,point_2):\n     return pd.Series(np.linalg.norm(df[[f\'{point_1}.x\',f\'{point_1}.y\']].values - df[[f\'{point_2}.x\',f\'{point_2}.y\']].values, axis =1 ))\n\n\nfolder = f"D:/big_setup/experiment_{experiment_no[0]}/concatenated_data_cam_mic_sync/"\nlabels_path = folder+"tracks/"\nfiles = glob.glob(labels_path+"*.csv")\nfor file in files:\n     temp = avg_euc_dist(pd.read_csv(file),\'body_2\',\'body_3\')\n     bins =  pd.qcut(range(20,40), 29,retbins = True,duplicates=\'drop\')[1]\n     temp_2 = pd.cut(temp, bins=bins, right=True, include_lowest=True)\n     bin_counts = temp.groupby(temp_2).size() \n\n     mode_bin = bin_counts.idxmax()\n     mode_count = bin_counts.max()\n     print(temp.mean())\n     print(mode_bin,mode_count)\n     # plt.figure()\n     # temp.hist(bins =bins)\n'

### Function to fill the skeleton head and nose points

In [138]:
DIST = {
    "nose_head": 40,
    "ear_head": 14,
    "head_neck": 10,
    "neck_body1": 20,
    "body1_body2": 22,
    "body2_body3": 22
}

def unit(v):
    n = np.linalg.norm(v)
    return v / n if n > 0 else None


def perp(v):
    return np.array([-v[1], v[0]])

def is_valid(p):
    return p is not None and not np.isnan(p).any()

def pixel_not_valid(p):
    if not(0<p[0]<1600) or not(0<p[1]<1400):
        return True
    else:
        return False


def find_spine_direction(P):
    """
    Returns a unit forward spine direction or None
    Forward = from head toward body
    Priority:
    1️⃣ body_1 ↔ body_2   (strongest)
    2️⃣ body_2 ↔ body_3
    3️⃣ neck ↔ body_1
    4️⃣ head ↔ body_1
    5️⃣ head ↔ neck
    6️⃣ ear perpendicular (weakest fallback)
    """


    # ---------------------------------------------------
    # perpendicular to ears
    # ---------------------------------------------------
    if is_valid(P["ear_l"]) and is_valid(P["ear_r"]):
        ear_vec = P["ear_r"] - P["ear_l"]
        d = unit(perp(ear_vec))
        if d is not None:
            return d
        
    # ---------------------------------------------------
    #  head → body_1
    # ---------------------------------------------------
    if is_valid(P["head"]) and is_valid(P["body_1"]):
        d = unit(P["body_1"] - P["head"])
        if d is not None:
            return d

    # ---------------------------------------------------
    # body_1 → body_2
    # ---------------------------------------------------
    if is_valid(P["body_1"]) and is_valid(P["body_2"]):
        d = unit(P["body_2"] - P["body_1"])
        if d is not None:
            return d

    # ---------------------------------------------------
    #  body_2 → body_3
    # ---------------------------------------------------
    if is_valid(P["body_2"]) and is_valid(P["body_3"]):
        d = unit(P["body_3"] - P["body_2"])
        if d is not None:
            return d
        
    # ---------------------------------------------------
    # head → neck
    # ---------------------------------------------------
    if is_valid(P["head"]) and is_valid(P["neck"]):
        d = unit(P["neck"] - P["head"])
        if d is not None:
            return d
        

    # ---------------------------------------------------
    # neck → body_1
    # ---------------------------------------------------
    if is_valid(P["neck"]) and is_valid(P["body_1"]):
        d = unit(P["body_1"] - P["neck"])
        if d is not None:
            return d





    return None

def fill_skeleton(P):

    # ---------------------------------------------------
    # HEAD
    # ---------------------------------------------------
    flag = 0 # this value keeps track of when the animal is rearing up, nose point is not extrapolated
    if np.isnan(P["head"]).any():

        # 1️⃣ Both ears
        if is_valid(P["ear_l"]) and is_valid(P["ear_r"]):
            P["head"] = (P["ear_l"] + P["ear_r"]) / 2
            if pixel_not_valid(P["head"]):
                P["head"] = np.array([np.nan,np.nan])
                
        # 3️⃣ Body_1 → body_2
        elif is_valid(P["body_1"]) and is_valid(P["body_2"]):
            d = unit(P["body_1"] - P["body_2"])
            if d is not None:
                neck = P["body_1"] - d * DIST["neck_body1"]
                P["head"] = neck - d * DIST["head_neck"]
                if pixel_not_valid(P["head"]):
                    neck = P["body_1"] - d * 5
                    P["head"] = neck - d * 5
                if pixel_not_valid(P["head"]):
                    P["head"] = np.array([np.nan,np.nan])

        # 2️⃣ Neck → body_1
        elif is_valid(P["neck"]) and is_valid(P["body_1"]):
            d = unit(P["neck"] - P["body_1"])
            if d is not None:
                P["head"] = P["neck"] - d * DIST["head_neck"]
                if pixel_not_valid(P["head"]):
                    P["head"] = P["neck"] - d * 5
                if pixel_not_valid(P["head"]):
                    P["head"] = np.array([np.nan,np.nan])

        # 4️⃣ One ear + neck
        elif (is_valid(P["ear_l"]) ^ is_valid(P["ear_r"])) and is_valid(P["neck"]):
            ear = P["ear_l"] if is_valid(P["ear_l"]) else P["ear_r"]
            d = unit(ear - P["neck"])
            if d is not None:
                P["head"] = ear - d * DIST["ear_head"]
                if pixel_not_valid(P["head"]):
                    P["head"] = ear - d * 5
                if pixel_not_valid(P["head"]):
                    P["head"] = np.array([np.nan,np.nan])

        # 5️⃣ One ear + nose - the animal is rearing up and none of the other points are there
        elif (is_valid(P["ear_l"]) ^ is_valid(P["ear_r"])) and is_valid(P["nose"]):
            flag = 1
            ear = P["ear_l"] if is_valid(P["ear_l"]) else P["ear_r"]
            d = unit(P["nose"] - ear)
            if d is not None:
                # place head slightly behind nose, toward skull center
                P["head"] = P["nose"] - d * (DIST["nose_head"]-20)
                if pixel_not_valid(P["head"]):
                    # fallback: midpoint
                    P["head"] = (P["nose"] + ear) / 2
                if pixel_not_valid(P["head"]):
                    P["head"] = np.array([np.nan, np.nan])


        # 6 Body_2 → body_3
        elif is_valid(P["body_2"]) and is_valid(P["body_3"]):
            d = unit(P["body_2"] - P["body_3"])
            if d is not None:
                body1 = P["body_2"] - d * DIST["body1_body2"]
                neck  = body1 - d * DIST["neck_body1"]
                P["head"] = neck - d * DIST["head_neck"]

                if pixel_not_valid(P["head"]):
                    body1 = P["body_2"] - d * 5
                    neck  = body1 - d * 5
                    P["head"] = neck - d * 5

                if pixel_not_valid(P["head"]):
                    P["head"] = np.array([np.nan, np.nan])


            
    # ---------------------------------------------------
    # SPINE DIRECTION (forward direction)
    # ---------------------------------------------------
    spine_dir = find_spine_direction(P)

    # ---------------------------------------------------
    # NOSE
    # ---------------------------------------------------
    if is_valid(P["head"]) and (spine_dir is not None) and (flag == 0):
        P["nose"] = P["head"] - spine_dir * DIST["nose_head"]
        if pixel_not_valid(P["nose"]):
            P["nose"] = P["head"] - spine_dir * 5
        if pixel_not_valid(P["head"]):
            P["nose"] = np.array([np.nan,np.nan])
    

    return P

    


In [139]:
# To fill the ear and body points    
# # ---------------------------------------------------
    # # EARS
    # # ---------------------------------------------------
    # if any(P["head"] is not np.nan) and spine_dir is not None:
    #     ear_axis = unit(perp(spine_dir)) * DIST["ear_head"]

    #     if P["ear_l"] is None:
    #         P["ear_l"] = P["head"] + ear_axis

    #     if P["ear_r"] is None:
    #         P["ear_r"] = P["head"] - ear_axis


    # # ---------------------------------------------------
    # # BODY
    # # ---------------------------------------------------
    # if spine_dir is not None:
    #     if P["body_1"] is None and any(P["neck"] is not np.nan):
    #         P["body_1"] = P["neck"] + spine_dir * DIST["neck_body1"]

    #     if P["body_2"] is None and P["body_1"] is not None:
    #         P["body_2"] = P["body_1"] + spine_dir * DIST["body1_body2"]

#### Code to iterate through experiments and create the corrected tracks

In [140]:
for exp in experiment_no:
    folder = f"D:/big_setup/experiment_{exp}/concatenated_data_cam_mic_sync/"
    track_folder = folder+"tracks/"
    corrected_tracks = folder+"corrected_tracks/"
    os.makedirs(corrected_tracks,exist_ok=True)

    labels = glob.glob(track_folder+"*.csv")
    for label in tqdm.tqdm(labels):
        preds = sio.load_csv(label)
        data = {}
        for frame in preds.labeled_frames:
            for instance in frame.instances:
                point_dict = {}
                scores = {}
                if len(data) == 0:
                    for entry in instance.points:
                        point_dict[f"{entry[-1]}"] = np.array([entry[0][0],entry[0][1]])
                        scores[f"{entry[-1]}"] = entry[1]

                    point_dict = fill_skeleton(point_dict)

                    data["track"] = [instance.track.name]
                
                    if instance.track.name not in ["track_35_mic","track_118_mic"] and not(any(i in label for i in excluded_indices)):
                        print(f"invalid track: {instance.track.name} for {label}")
                        # raise Exception(f"invalid track: {instance.track.name}")
                    data["frame_idx"] = [frame.frame_idx]
                    data["instance.score"] = [instance.score]

                    for i,v in point_dict.items():
                        data[f"{i}.x"] = [v[0]]
                        data[f"{i}.y"] = [v[1]]
                        data[f"{i}.score"] = [scores[i]]
                else:
                    for entry in instance.points:
                        point_dict[f"{entry[-1]}"] = np.array([entry[0][0],entry[0][1]])
                        scores[f"{entry[-1]}"] = entry[1]

                    point_dict = fill_skeleton(point_dict)

                    data["track"].append(instance.track.name)
                    if instance.track.name not in ["track_35_mic","track_118_mic"] and not(any(i in label for i in excluded_indices)):
                        # raise Exception(f"invalid track: {instance.track.name}")
                        print(f"invalid track: {instance.track.name} for {label}")
                    data["frame_idx"].append(frame.frame_idx)
                    data["instance.score"].append(instance.score)

                    for i,v in point_dict.items():
                        data[f"{i}.x"].append(v[0])
                        data[f"{i}.y"].append(v[1])
                        data[f"{i}.score"].append(scores[i])

        df = pd.DataFrame.from_dict(data)

        tmp_str = ".".join(label.split("\\")[-1].split(".")[:-1])+".corrected.csv"
        df.to_csv(corrected_tracks+tmp_str,index=False)

    print(f"Experiment {exp} done")


 14%|█▍        | 1/7 [00:00<00:02,  2.77it/s]

invalid track: track_0 for D:/big_setup/experiment_384/concatenated_data_cam_mic_sync/tracks\labels.body_1_anchor.video_gily_center_000.analysis.csv
invalid track: track_0 for D:/big_setup/experiment_384/concatenated_data_cam_mic_sync/tracks\labels.body_1_anchor.video_gily_center_000.analysis.csv
invalid track: track_0 for D:/big_setup/experiment_384/concatenated_data_cam_mic_sync/tracks\labels.body_1_anchor.video_gily_center_000.analysis.csv
invalid track: track_0 for D:/big_setup/experiment_384/concatenated_data_cam_mic_sync/tracks\labels.body_1_anchor.video_gily_center_000.analysis.csv
invalid track: track_0 for D:/big_setup/experiment_384/concatenated_data_cam_mic_sync/tracks\labels.body_1_anchor.video_gily_center_000.analysis.csv
invalid track: track_0 for D:/big_setup/experiment_384/concatenated_data_cam_mic_sync/tracks\labels.body_1_anchor.video_gily_center_000.analysis.csv
invalid track: track_0 for D:/big_setup/experiment_384/concatenated_data_cam_mic_sync/tracks\labels.body_1

100%|██████████| 7/7 [00:05<00:00,  1.24it/s]

Experiment 384 done


#### Code to plot the points on the videos for testing

In [141]:
# testing the labels by exporting csv to sleap - didnt work, sleap throws an error - need the metadata
# for exp in experiment_no:
#     folder = f"D:/big_setup/experiment_{exp}/concatenated_data_cam_mic_sync/"
#     corrected_tracks = folder+"corrected_tracks/"
#     #os.makedirs(corrected_tracks,exist_ok=True)

#     labels = glob.glob(corrected_tracks+"*.csv")

#     for label in tqdm.tqdm(labels):
#         preds = sio.load_csv(label)
#         preds.save(".".join(label.split(".")[:-1])+".slp")


In [142]:
# import cv2
# import matplotlib.pyplot as plt
# import tqdm


# # Skeleton connections (bones)
# skeleton = [
#     ("nose", "head"),
#     ("nose", "ear_l"),
#     ("nose", "ear_r"),
#     ("head", "neck"),
#     ("neck", "body_1"),
#     ("body_1", "body_2"),
#     ("body_2", "body_3"),
#     ("body_3", "tail_base")
# ]

# def is_valid(p):
#     return p is not None and not np.isnan(p).any()

# for exp in experiment_no:
#     folder = f"D:/big_setup/experiment_{exp}/concatenated_data_cam_mic_sync/"
#     corrected_tracks = folder+"corrected_tracks/"
#     #os.makedirs(corrected_tracks,exist_ok=True)

#     labels = glob.glob(corrected_tracks+"*.csv")
#     labels.sort(key=natural_keys)
#     vids = glob.glob(folder+"*.mp4")
#     vids.sort(key=natural_keys)


#     for label,vid in tqdm.tqdm(zip(labels,vids)):
#         data = pd.read_csv(label)
        

#         keypoints = pd.DataFrame.from_dict({
#             "frame": data["frame_idx"],
#             "nose": np.array(zip(data["nose.x"],data["nose.y"])),
#             "head": np.array(zip(data["head.x"],data["head.y"])),
#             "ear_l": np.array(zip(data["ear_l.x"],data["ear_l.y"])),
#             "ear_r": np.array(zip(data["ear_r.x"],data["ear_r.y"])),
#             "neck": np.array(zip(data["neck.x"],data["neck.y"])),
#             "body_1": np.array(zip(data["body_1.x"],data["body_1.y"])),
#             "body_2": np.array(zip(data["body_2.x"],data["body_2.y"])),
#             "body_3": np.array(zip(data["body_3.x"],data["body_3.y"])),
#             "tail_base": np.array(zip(data["tail_base.x"],data["tail_base.y"]))})
        

#         cap = cv2.VideoCapture(vid)

        
#         # Get video properties
#         fps = cap.get(cv2.CAP_PROP_FPS)
#         width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
#         height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

#         # Define video writer
#         fourcc = cv2.VideoWriter_fourcc(*"mp4v")
#         out = cv2.VideoWriter(
#             vid.split("\\")[0]+"\\"+vid.split("\\")[-1].split(".")[0]+"_w_track.mp4",
#             fourcc,
#             fps,
#             (width, height)
#         )

#         if not cap.isOpened():
#             print("Error: Cannot open video")
#             exit()

#         frame_no = 0

#         while True:
#             ret, frame = cap.read()
#             if not ret:
#                 break  # end of video
#             joints = set()
#             for idx,row in keypoints[keypoints["frame"] == frame_no].iterrows():
#                 for joint1, joint2 in skeleton:
#                     if is_valid(row[joint1]) and is_valid(row[joint2]):
#                         x1, y1 = row[joint1]
#                         x2, y2 = row[joint2]
#                         if any([x1,x2,y1,y2]) < 0.0:
#                             continue
#                         joints.add((x1,y1))
#                         joints.add((x2,y2))
#                         cv2.line(frame, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 2)
#                 for x,y in joints:
#                         cv2.circle(frame, (int(x), int(y)), 5, (255, 0, 0), -1)

#             out.write(frame)  # write frame to v
#             frame_no+=1

#         out.release()
#         cap.release()
#         cv2.destroyAllWindows()


In [143]:
# # deleting the files after proof reading
# folder = f"D:/big_setup/experiment_{experiment_no[0]}/concatenated_data_cam_mic_sync/"
# vids = glob.glob(folder+"*_w_track.mp4")
# for i in vids:
#     os.remove(i) 